In [5]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-6"

In [6]:
def add_user_message(messages, text):
    user_message={"role":"user","content":text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message={"role":"assistant","content":text}
    messages.append(assistant_message)

def chat(messages, system=None):
    params={
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

# 시스템 메시지가 설정된 경우에만 params에 system을 포함하게끔 하는 if 함수
    if system:
        params["system"]= system
# (**)는 params 주머니 안의 내용물을 자동으로 꺼내서 API 양식에 맞게 채워주는 규칙임     
    message=client.messages.create(**params)
    return message.content[0].text

In [8]:
# 실습문제2. 시스템 프롬프트를 통해 토큰을 최소화하며 질문에 간결하게 답하기 
# - 프롬프트 설정 하지 않은 경우
messages = []

add_user_message(
    messages,
    "기말 현재 차입금 상환약정을 위반했는데, 은행에서 해당 차입금에 대한 상환요청이 오지 않았다는 이유로 회사는 비유동 유지하고자 희망하는 상황. 해당 차입금에 대해 실증절차 수행 후 감사조서화 필요.")
answer = chat(messages)
answer

'# 차입금 상환약정 위반 관련 감사절차 및 조서화\n\n## I. 관련 기준 검토\n\n### K-IFRS 제1001호 (재무제표 표시)\n\n> **문단 74-75 핵심 요약**\n> - 기말 현재 약정 위반 시 → **즉시 유동 분류 원칙**\n> - 단, 보고기간 후 12개월 이상 결제를 연기할 수 있는 **확약(Waiver)을 대여자가 제공**한 경우 → 비유동 유지 가능\n> - 단순히 "상환요청이 없었다"는 사실만으로는 비유동 분류 **불충분**\n\n---\n\n## II. 실증절차 수행 내역\n\n### 1️⃣ 차입금 약정서 검토\n\n| 절차 | 내용 | 결과 |\n|------|------|------|\n| 차입약정서 입수 및 정독 | 상환약정 조항 (Covenant) 확인 | 위반된 조항 특정 |\n| 위반 조항 유형 확인 | 재무비율 / 담보유지 / 기한이익상실 조항 등 | ○○조항 위반 확인 |\n| 기한이익상실(Acceleration) 조항 | 위반 시 즉시 상환요구 가능 여부 | **즉시 상환요구 가능** 확인 |\n\n> 📌 **핵심포인트**: 약정 위반 시 대여자가 즉시 상환을 요구할 **권리**가 발생했는지 여부\n\n---\n\n### 2️⃣ 은행 조회서 발송 및 회수\n\n```\n[조회 항목]\n① 기말 현재 차입금 잔액 확인\n② 약정 위반 사실 인지 여부\n③ 위반에 대한 면제(Waiver) 제공 여부\n④ 상환유예 확약서(Waiver Letter) 발급 여부\n⑤ 보고기간 후 현재까지 상환청구 의사 여부\n```\n\n| 조회 결과 | 감사 판단 |\n|-----------|-----------|\n| Waiver 확약서 수령 | 비유동 분류 가능 (조건 충족 시) |\n| **Waiver 없음 / 무응답** | **유동 분류 필요** |\n| 상환청구 없다는 단순 확인서 | **유동 분류 필요** (기준 불충족) |\n\n---\n\n### 3️⃣ 경영진 조회 및 질문\n\n- [ ] 약정 위반 인지 시점 확인\

In [7]:
# 실습문제2. 시스템 프롬프트를 통해 토큰을 최소화하며 질문에 간결하게 답하기 
# - 프롬프트 설정 한 경우 
messages = []

add_user_message(
    messages,
    "기말 현재 차입금 상환약정을 위반했는데, 은행에서 해당 차입금에 대한 상환요청이 오지 않았다는 이유로 회사는 비유동 유지하고자 희망하는 상황. 해당 차입금에 대해 실증절차 수행 후 감사조서화 필요.")
answer = chat(messages, system="너는 10년 차 감사인이야. 설명은 생략하고 '수행절차' '필수 추가검토자료' '결론'과 '조서용 문구'만 아주 간결하게 답변해.")

answer

'## 차입금 분류 검토 (기말 상환약정 위반)\n\n---\n\n### 수행절차\n\n1. 차입약정서 원본 입수 → 위반 조항 및 기한이익상실 조항 확인\n2. 위반 사실 발생일자 및 내용 특정\n3. **은행 확인서** 징구 → 기한이익상실 권리 보유 여부 및 유예(waiver) 제공 여부 확인\n4. 기말일 현재 은행이 **공식적·문서화된 유예(waiver)** 를 제공했는지 여부 확인\n5. 유예 존재 시 → 유예 기간이 기말일로부터 **최소 12개월 초과** 여부 확인\n6. 후속사건 검토 → 보고서일 현재까지 실제 상환 청구 여부 확인\n\n---\n\n### 필수 추가검토자료\n\n| 자료 | 목적 |\n|------|------|\n| 차입약정서 원본 | 기한이익상실 조항 확인 |\n| 은행 확인서 (외부) | waiver 공식 제공 여부 확인 |\n| Waiver letter (있는 경우) | 유예 기간·조건 확인 |\n| 이사회 의사록 | 위반 인지 및 대응 현황 확인 |\n| 재무제표 주석 초안 | 공시 적정성 확인 |\n\n---\n\n### 결론\n\n> **K-IFRS 1001.74**: 기말일 현재 약정 위반 시, 대여자가 기말일 이전에 기한이익상실을 **최소 12개월 이상 유예하기로 합의**하지 않은 한 → **유동 분류 필수**\n\n- 은행의 단순 미청구는 유예(waiver)에 해당하지 않음\n- Waiver letter 없거나 유예기간 12개월 미만 → **유동 재분류 필요**\n- 회사가 수정 거부 시 → **한정의견 검토**\n\n---\n\n### 조서용 문구\n\n> 기말일 현재 차입금(XXX원)에 대한 상환약정 위반이 확인되었으며, 은행으로부터 입수한 외부확인서 및 관련 약정서 검토 결과, 대여자가 기말일 이전에 동 위반에 대해 K-IFRS 제1001호 문단 74에서 요구하는 **12개월 이상의 공식적 유예(waiver)를 제공한 사실이 없음**을 확인하였다. 따라서 동 차입금은 유동부채로 재분류되어야 하며, 회사의 